# Reproduced vs original score correlations

This notebook is a runnable wrapper around `scripts/correlate_original_reproduced.py`. Set `REPRODUCED_DIR` below to the reproduced NLA folder you want to correlate against `data/original/`, then run all cells to regenerate and inspect the outputs.

The reproduced measures use raw numerators divided by `nr_of_words`: `risk / nr_of_words` for PRisk and PRiskT, and `sentiment / nr_of_words` for PSentiment.


In [ ]:
from __future__ import annotations

import csv
import math
import subprocess
import sys
from html import escape
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "scripts" / "correlate_original_reproduced.py").exists():
            return path
    raise RuntimeError("Could not find the prisk-replication repository root.")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPT = REPO_ROOT / "scripts" / "correlate_original_reproduced.py"
CORRELATIONS_DIR = REPO_ROOT / "data" / "correlations"
REPRODUCED_DIR = REPO_ROOT / "data" / "reproduced"
REPRODUCED_DIR


## Regenerate Correlations


In [ ]:
result = subprocess.run(
    [sys.executable, str(SCRIPT), "--reproduced-dir", str(REPRODUCED_DIR)],
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()


In [ ]:
def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open(newline="") as file:
        return list(csv.DictReader(file))


def as_number(value: str) -> float | None:
    try:
        parsed = float(value)
    except (TypeError, ValueError):
        return None
    if not math.isfinite(parsed):
        return None
    return parsed


def format_value(value: object) -> str:
    if value is None:
        return ""
    if isinstance(value, float):
        return f"{value:.6f}"
    text = str(value)
    parsed = as_number(text)
    if parsed is not None and "." in text:
        return f"{parsed:.6f}"
    return text


def display_table(rows: list[dict[str, object]]):
    try:
        from IPython.display import HTML
    except Exception:
        return rows

    if not rows:
        return HTML("<em>No rows</em>")

    columns = list(rows[0].keys())
    header = "".join(f"<th>{escape(column)}</th>" for column in columns)
    body_rows = []
    for row in rows:
        cells = "".join(
            f"<td>{escape(format_value(row.get(column)))}</td>"
            for column in columns
        )
        body_rows.append(f"<tr>{cells}</tr>")
    return HTML(
        "<table>"
        "<thead><tr>" + header + "</tr></thead>"
        "<tbody>" + "".join(body_rows) + "</tbody>"
        "</table>"
    )


## Generated Files


In [ ]:
generated_files = [
    "firm_quarter_correlations.csv",
    "quarterly_mean_correlations.csv",
]

display_table(
    [
        {
            "file": name,
            "exists": (CORRELATIONS_DIR / name).exists(),
            "size_bytes": (CORRELATIONS_DIR / name).stat().st_size if (CORRELATIONS_DIR / name).exists() else "",
        }
        for name in generated_files
    ]
)


## gvkey-dateQ Correlations


In [ ]:
gvkey_dateq_correlations = read_csv(
    CORRELATIONS_DIR / "firm_quarter_correlations.csv"
)
display_table(gvkey_dateq_correlations)


## dateQ Correlations


In [ ]:
dateq_correlations = read_csv(
    CORRELATIONS_DIR / "quarterly_mean_correlations.csv"
)
display_table(dateq_correlations)
